# DSBA6010 Final Project
## LLM-Based Privacy Attacks on Employee Data

**Dataset:** [IBM HR Analytics Employee Attrition & Performance](https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset)  
**Project Overview:**  
This notebook investigates the privacy risks of sharing employee data in the age of Large Language Models (LLMs). We:
1. Load and explore the IBM HR Analytics dataset
2. Apply Differential Privacy (DP) at multiple epsilon levels
3. Assess utility (model accuracy) after privatization
4. Attack the privatized dataset using Membership Inference and Attribute Inference attacks

**References:**
- Liu et al. (2025) — Evaluating LLM-based Personal Information Extraction and Countermeasures. USENIX Security.
- Staab et al. (2024) — Beyond Memorization: Violating Privacy via Inference with LLMs. ICLR.
- Jayaraman & Evans (2022) — Are Attribute Inference Attacks Just Imputation? CCS.
- Kandpal et al. (2024) — User Inference Attacks on Large Language Models. EMNLP.
- Cirillo et al. (2025) — Augmenting Anonymized Data with AI. arXiv.
- Yan et al. (2025) — Stop Tracking Me! Proactive Defense Against Attribute Inference Attacks. arXiv.

## 0. Install Dependencies

In [ ]:
# Run this cell once to install required packages
!pip install pandas numpy matplotlib seaborn scikit-learn diffprivlib kaggle

## 1. Setup & Data Loading

**Option A:** Download via Kaggle API (recommended)  
**Option B:** Manual download — go to the [Kaggle page](https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset), download `WA_Fn-UseC_-HR-Employee-Attrition.csv`, and place it in the same folder as this notebook.

In [ ]:
# Option A: Kaggle API download
# Requires ~/.kaggle/kaggle.json with your API credentials
# Get your API key from: https://www.kaggle.com/settings -> API -> Create New Token

import os

DATASET = "pavansubhasht/ibm-hr-analytics-attrition-dataset"
DATA_FILE = "WA_Fn-UseC_-HR-Employee-Attrition.csv"

if not os.path.exists(DATA_FILE):
    print("Downloading dataset from Kaggle...")
    os.system(f"kaggle datasets download -d {DATASET} --unzip")
    print("Download complete.")
else:
    print(f"Dataset already exists: {DATA_FILE}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset
df = pd.read_csv(DATA_FILE)

print(f"Dataset shape: {df.shape}")
print(f"\nColumns ({len(df.columns)}):")
print(list(df.columns))

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Basic info
df.info()

In [ ]:
# Summary statistics for numeric columns
df.describe().T

In [ ]:
# Check for missing values
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.sum() > 0 else "None — dataset is complete.")

In [ ]:
# Target variable: Attrition
attrition_counts = df['Attrition'].value_counts()
print("Attrition distribution:")
print(attrition_counts)
print(f"\nAttrition rate: {attrition_counts['Yes'] / len(df):.1%}")

fig, ax = plt.subplots(figsize=(5, 4))
attrition_counts.plot(kind='bar', color=['steelblue', 'salmon'], ax=ax)
ax.set_title('Attrition Distribution')
ax.set_xlabel('Attrition')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of key numeric features
numeric_cols = ['Age', 'MonthlyIncome', 'YearsAtCompany', 'DistanceFromHome',
                'JobSatisfaction', 'WorkLifeBalance', 'PerformanceRating']

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col], bins=20, color='steelblue', edgecolor='white')
    axes[i].set_title(col)
    axes[i].set_ylabel('Count')

# Hide unused subplot
axes[-1].set_visible(False)
plt.suptitle('Distribution of Key Features', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Attrition by Department and Gender
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

dept_attrition = df.groupby('Department')['Attrition'].apply(
    lambda x: (x == 'Yes').mean() * 100
).sort_values(ascending=False)
dept_attrition.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Attrition Rate by Department (%)')
axes[0].set_ylabel('Attrition Rate (%)')
axes[0].tick_params(axis='x', rotation=15)

gender_attrition = df.groupby('Gender')['Attrition'].apply(
    lambda x: (x == 'Yes').mean() * 100
)
gender_attrition.plot(kind='bar', ax=axes[1], color='salmon')
axes[1].set_title('Attrition Rate by Gender (%)')
axes[1].set_ylabel('Attrition Rate (%)')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (numeric features only)
numeric_df = df.select_dtypes(include=[np.number])

plt.figure(figsize=(14, 10))
corr = numeric_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))  # hide upper triangle
sns.heatmap(corr, mask=mask, annot=False, cmap='coolwarm', center=0,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Preprocessing

Encode categorical variables and split into train/test sets.  
This preprocessed dataset will serve as the **baseline** before any privatization.

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

df_processed = df.copy()

# Drop columns with zero variance (constants in this dataset)
constant_cols = ['EmployeeCount', 'Over18', 'StandardHours']
df_processed.drop(columns=constant_cols, inplace=True)
print(f"Dropped constant columns: {constant_cols}")

# Encode binary/categorical columns
le = LabelEncoder()
categorical_cols = df_processed.select_dtypes(include='object').columns.tolist()
print(f"\nEncoding categorical columns: {categorical_cols}")

for col in categorical_cols:
    df_processed[col] = le.fit_transform(df_processed[col])

# Features and target
X = df_processed.drop(columns=['Attrition'])
y = df_processed['Attrition']  # 0 = No, 1 = Yes

# Train/test split (stratified to preserve attrition ratio)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nTrain size: {X_train.shape[0]} | Test size: {X_test.shape[0]}")
print(f"Features: {X_train.shape[1]}")
print(f"Attrition rate (train): {y_train.mean():.1%}")

## 4. Baseline Model (No Privacy)

Train a Logistic Regression classifier on the original data.  
This will serve as the **accuracy ceiling** to compare against privatized versions.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

baseline_model = LogisticRegression(max_iter=1000, random_state=42)
baseline_model.fit(X_train_scaled, y_train)

y_pred = baseline_model.predict(X_test_scaled)
y_prob = baseline_model.predict_proba(X_test_scaled)[:, 1]

print("=== Baseline Model (No Privacy) ===")
print(classification_report(y_test, y_pred, target_names=['No Attrition', 'Attrition']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")

---
## Next Steps

- **Section 5:** Apply Differential Privacy using `diffprivlib` at ε = 0.1, 1.0, 10.0
- **Section 6:** Compare utility (accuracy/AUC) across epsilon levels
- **Section 7:** Membership Inference Attack
- **Section 8:** Attribute Inference Attack